In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [5]:
## Load the dataset. 
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [7]:
# Encode categorical variables.
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])
data 




,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [8]:
## One-hot encode the 'Geography' column.
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder()
geography_encoded = one_hot_encoder_geo.fit_transform(data[['Geography']])
geography_encoded


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [9]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoded_df = pd.DataFrame(geography_encoded.toarray(), columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

In [11]:
## combine the one-hot encoded 'Geography' columns with the original dataset and drop the original 'Geography' column.
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [12]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [13]:
## Save the encoders and scaler for later use.
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

with open('one_hot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(one_hot_encoder_geo, f)

    

In [14]:
## Divide the dataset into features and target variable.
X = data.drop('Exited', axis=1)
y = data['Exited']
## Split the dataset into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scale these features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Save the scaler for later use.
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


## ANN Implementation


In [15]:

import torch
import torch.nn as nn
import torch.optim as optim

## Define the neural network architecture. -> ANN model.
model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 64), # input layer -> 64 neurons
    nn.ReLU(),
    nn.Linear(64, 32),                # hidden layer -> 32 neurons  
    nn.ReLU(),
    nn.Linear(32, 1),                  # output layer -> 1 neuron   
    nn.Sigmoid()                       # sigmoid activation for the binary classification output.

)

# Loss function and optimizer.
criterion = nn.BCELoss()  # Binary Cross Entropy Loss for binary classification.
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer with a learning rate of 0.001.

print("Model Architecture:", model)


Model Architecture: Sequential(
  (0): Linear(in_features=12, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
  (5): Sigmoid()
)


In [16]:

from torch.utils.tensorboard import SummaryWriter

# TensorBoard writer
writer = SummaryWriter(log_dir="logs/fit")

# Early stopping parameters
patience = 10
best_loss = float("inf")
epochs_no_improve = 0
n_epochs = 100

# Convert data to tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

# Validation split
val_size = int(0.2 * len(X_train_tensor))
X_val, y_val = X_train_tensor[:val_size], y_train_tensor[:val_size]
X_train, y_train = X_train_tensor[val_size:], y_train_tensor[val_size:]

for epoch in range(n_epochs):
    # Training
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)

    # Log to TensorBoard
    writer.add_scalar("Loss/train", loss.item(), epoch)
    writer.add_scalar("Loss/val", val_loss.item(), epoch)

    print(f"Epoch {epoch+1}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

    # Early stopping check
    if val_loss.item() < best_loss:
        best_loss = val_loss.item()
        epochs_no_improve = 0
        best_model_state = model.state_dict()  # save best weights
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered!")
            model.load_state_dict(best_model_state)  # restore best weights
            break

writer.close()


Epoch 1, Train Loss: 0.6452, Val Loss: 0.6378
Epoch 2, Train Loss: 0.6365, Val Loss: 0.6292
Epoch 3, Train Loss: 0.6279, Val Loss: 0.6209
Epoch 4, Train Loss: 0.6196, Val Loss: 0.6128
Epoch 5, Train Loss: 0.6116, Val Loss: 0.6050
Epoch 6, Train Loss: 0.6038, Val Loss: 0.5974
Epoch 7, Train Loss: 0.5962, Val Loss: 0.5901
Epoch 8, Train Loss: 0.5889, Val Loss: 0.5830
Epoch 9, Train Loss: 0.5818, Val Loss: 0.5761
Epoch 10, Train Loss: 0.5749, Val Loss: 0.5695
Epoch 11, Train Loss: 0.5683, Val Loss: 0.5630
Epoch 12, Train Loss: 0.5618, Val Loss: 0.5569
Epoch 13, Train Loss: 0.5557, Val Loss: 0.5510
Epoch 14, Train Loss: 0.5497, Val Loss: 0.5453
Epoch 15, Train Loss: 0.5441, Val Loss: 0.5399
Epoch 16, Train Loss: 0.5387, Val Loss: 0.5348
Epoch 17, Train Loss: 0.5335, Val Loss: 0.5300
Epoch 18, Train Loss: 0.5287, Val Loss: 0.5255
Epoch 19, Train Loss: 0.5241, Val Loss: 0.5212
Epoch 20, Train Loss: 0.5199, Val Loss: 0.5173
Epoch 21, Train Loss: 0.5159, Val Loss: 0.5136
Epoch 22, Train Loss: 

In [17]:
torch.save(model.state_dict(), "ann_model.pth")
torch.save(model, "ann_model_full.pth")  # Save the entire model for later use.


In [22]:
## Load the TensorBoard extension in Jupyter Notebook and visualize the training process.
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [24]:
  %reload_ext tensorboard

In [23]:
%tensorboard --logdir logs/fit


Reusing TensorBoard on port 6006 (pid 20724), started 1:27:10 ago. (Use '!kill 20724' to kill it.)

In [25]:
import pickle

# Save state dict (recommended)
with open("model_state.pkl", "wb") as f:
    pickle.dump(model.state_dict(), f)

# Or save full model (less portable)
with open("model_full.pkl", "wb") as f:
    pickle.dump(model, f)


In [26]:
import torch
import pickle

## Load the trained model, scaler pickle, and encoder pickles for inference.
model = torch.load("ann_model_full.pth", weights_only=False)

## Load the label encoder, one-hot encoder, and scaler for inference.
with open("label_encoder.pkl", "rb") as file:
    label_encoder = pickle.load(file)

with open("one_hot_encoder_geo.pkl", "rb") as file:
    one_hot_encoder_geo = pickle.load(file)

with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)


In [27]:
## Example input data for inference.
example_input = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,    
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [28]:
# One-hot encode the 'Geography' column for the example input.
# NOTE: the original code mistakenly used `label_encoder` (fit on the 'Gender'
# column) to transform 'Geography'. We must use `one_hot_encoder_geo` instead.
# We wrap the value in a DataFrame with the correct column name (rather than a
# bare list) so sklearn doesn't warn about missing feature names.
geo_input = pd.DataFrame({'Geography': [example_input['Geography']]})
geo_encoded = one_hot_encoder_geo.transform(geo_input).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [29]:
## Build the full input row: encode Gender, merge one-hot Geography columns,
## and align column order with what the scaler was fit on.
input_df = pd.DataFrame([example_input])
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)

## Reindex columns to match the exact order the scaler saw during training
## (StandardScaler stores this automatically as `feature_names_in_` when
## it was fit on a pandas DataFrame).
input_df = input_df[scaler.feature_names_in_]

input_scaled = scaler.transform(input_df)
input_scaled


array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [30]:
## Run the scaled input through the trained model to get a churn prediction.
model.eval()
with torch.no_grad():
    input_tensor = torch.tensor(input_scaled, dtype=torch.float32)
    prediction = model(input_tensor)
    prediction_prob = prediction.item()

print(f"Churn Probability: {prediction_prob:.4f}")
if prediction_prob > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")


Churn Probability: 0.0746
The customer is unlikely to churn.
